# 04 · Desafio 2 — Quando a fauna está mais ativa?
**Pergunta:** em quais períodos do dia há mais registros de animais?
Escolha três espécies e compare seus horários de maior ocorrência.

Períodos: 
 · Madrugada (0–5h)
 · Manhã (6–11h)
 · Tarde (12–17h)
 · Noite (18–23h)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

silver = spark.table("workspace.fauna.silver_registros")
total = silver.count()

periodos = ["Madrugada", "Manhã", "Tarde", "Noite"]

In [0]:
ordem = F.array_position(F.array(*[F.lit(p) for p in periodos]), F.col("periodo_dia"))

por_periodo = (
    silver.groupBy("periodo_dia")
    .count()
    .withColumn("percentual", F.round(F.col("count") / total * 100, 1))
    .orderBy(ordem)
)
display(por_periodo)

In [0]:
por_hora = silver.groupBy("hora").count().orderBy("hora")
display(por_hora)

In [0]:
import matplotlib.pyplot as plt
from pathlib import Path

pasta_imagens = Path.cwd().parent / "imagens"
pasta_imagens.mkdir(exist_ok=True)

cor_faixa = "#dcdad3"

pdf_hora = por_hora.toPandas()

fig, ax = plt.subplots(figsize=(10, 4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.axvspan(-0.5, 5.5, color=cor_faixa, zorder=0)    # faixa da madrugada
ax.axvspan(17.5, 23.5, color=cor_faixa, zorder=0)   # faixa da noite

ax.bar(pdf_hora["hora"], pdf_hora["count"], width=0.8,
       color="#2a78d6", edgecolor="#fcfcfb", linewidth=2, zorder=2)

ax.set_title("A quantidade de registro aumenta no inicio da manhã e noite", loc="left", color="#0b0b0b", fontsize=12)
ax.set_xlabel("Hora do dia (faixas cinza = madrugada e noite)", color="#52514e")
ax.set_ylabel("Quantidade de registros", color="#52514e")
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f"{h}h" for h in range(0, 24, 2)])
ax.set_xlim(-0.5, 23.5)
ax.tick_params(colors="#52514e")
ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=1)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")

plt.tight_layout()
fig.savefig(pasta_imagens / "desafio2_registros_por_hora.png", dpi=150)
plt.show()

In [0]:
pivot = (
    silver.groupBy("especie")
    .pivot("periodo_dia", periodos)
    .count()
    .fillna(0)
)
total_especie = sum(F.col(p) for p in periodos)

perfil = pivot.select(
    "especie",
    *[F.round(F.col(p) / total_especie * 100, 1).alias(f"{p} (%)") for p in periodos],
    total_especie.alias("registros"),
)
display(perfil.orderBy(F.desc("Noite (%)")))

In [0]:
especies_escolhidas = ["Sapo-cururu", "Perereca-verde", "Rã-manteiga"]

validas = [linha.especie for linha in silver.select("especie").distinct().collect()]
assert all(e in validas for e in especies_escolhidas), f"Confira os nomes. Válidos: {sorted(validas)}"

In [0]:
janela_especie = Window.partitionBy("especie")

hora_especie = (
    silver.filter(F.col("especie").isin(especies_escolhidas))
    .groupBy("especie", "hora")
    .count()
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(janela_especie) * 100, 1))
)

In [0]:
ranking = Window.partitionBy("especie").orderBy(F.desc("count"))

picos = (
    hora_especie
    .withColumn("posicao", F.row_number().over(ranking))
    .filter(F.col("posicao") <= 3)
    .orderBy("especie", "posicao")
)
display(picos)

In [0]:
pdf = hora_especie.toPandas()
cores = ["#2a78d6", "#eb6834", "#1baf7a"]

fig, ax = plt.subplots(figsize=(11, 4.5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.axvspan(-0.5, 5.5, color="#f0efec", zorder=0)
ax.axvspan(17.5, 23.5, color="#f0efec", zorder=0)

for especie, cor in zip(especies_escolhidas, cores):
    serie = (pdf[pdf["especie"] == especie]
             .set_index("hora")["pct"]
             .reindex(range(24), fill_value=0)) # garante as 24 horas, mesmo sem registro
    ax.plot(serie.index, serie.values, color=cor, linewidth=2,
            marker="o", markersize=5, label=especie, zorder=3)
    hora_pico = int(serie.idxmax())
    ax.annotate(f"pico {hora_pico}h", xy=(hora_pico, serie.max()),
                xytext=(0, 8), textcoords="offset points",
                ha="center", color="#52514e", fontsize=9)

ax.set_title("Os 3 anfíbios apresentam hábito noturno", loc="left", color="#0b0b0b", fontsize=12)
ax.set_xlabel("Hora do dia (faixas cinza = madrugada e noite)", color="#52514e")
ax.set_ylabel("% dos registros da espécie", color="#52514e")
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f"{h}h" for h in range(0, 24, 2)])
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, pdf["pct"].max() * 1.25)             # espaço para os rótulos de pico
ax.tick_params(colors="#52514e")
ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=1)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1), labelcolor="#52514e")

plt.tight_layout()
fig.savefig(pasta_imagens / "desafio2_horarios_especies.png", dpi=150, bbox_inches="tight")
plt.show()

### Conclusão
Períodos do dia (todas as espécies):
- O período com mais registros é de manhã (28,4%), seguido da tarde (26,3%).
- Olhando hora a hora, os picos acontecem em 6h00 e 18h00.

**Comparação das 3 espécies:**
| Espécie | Perfil | Horários de pico | % no(s) período(s) principal(is) |
|---|---|---|---|
| Sapo-cururu | Noturno | 1h00 | O período com mais registros é de madrugada (44,3%), seguido da noite (37,1%) |
| Perereca-verde | Noturno | 19h00 | O período com mais registros é de noite (47,6%), seguido da madrugada (34,2%) |
| Rã-manteiga | Noturno | 18h00 | O período com mais registros é de madrugada (42%), seguido da noite (36,3%) |

**Resposta:** embora as três espécies apresentem hábitos predominantemente noturnos, existem diferenças nos horários de maior ocorrência. O sapo-cururu se destaca na madrugada, a perereca-verde apresenta maior concentração à noite e a rã-manteiga demonstra dois picos de atividade, um na madrugada e outro no início da noite.
Esses padrões sugerem que o período de transição entre o dia e a noite, além da madrugada, são momentos importantes para a atividade desses anfíbios.